# Ball detector training on Colab (T4)

**Before running:** upload `tennis_ball_dataset_colab.zip` (built locally, ~925MB - contains `data/raw/train`, `data/raw/valid`, the current `weights/ball_detector.pt` checkpoint, and `configs/ball_dataset.yaml`) to your Google Drive at:

`My Drive/tennis_ball_dataset_colab.zip`

Then: Runtime -> Change runtime type -> T4 GPU, and run the cells in order.

Checkpoints save straight to Drive during training (`--project` points at Drive, not local Colab disk), so if the runtime disconnects mid-run (free tier: ~90min idle timeout, ~12h hard cap), progress up to the last saved epoch is not lost - you'd just re-run the training cell pointing `--model` at the saved `last.pt` to keep fine-tuning from there (this restarts the epoch/LR schedule rather than a true resume, but is fine for further fine-tuning).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the dataset bundle from Drive into local Colab disk (fast local reads during training)
!mkdir -p /content/data_bundle
!unzip -q "/content/drive/MyDrive/tennis_ball_dataset_colab.zip" -d /content/data_bundle
!find /content/data_bundle -maxdepth 3 -type d

In [ ]:
# Clone the repo (public GitHub) at the working branch
!git clone -b claude/tennis-ball-yolo-tracking-p8ntwh --depth 1 https://github.com/will-wang1/tennis-tracking-yolo-v1.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# Drop the unzipped dataset/checkpoint/config into the repo's expected layout
!rm -rf /content/repo/data/raw/train /content/repo/data/raw/valid
!mkdir -p /content/repo/data/raw
!cp -r /content/data_bundle/data/raw/train /content/repo/data/raw/train
!cp -r /content/data_bundle/data/raw/valid /content/repo/data/raw/valid
!cp /content/data_bundle/weights/ball_detector.pt /content/repo/weights/ball_detector.pt
!cp /content/data_bundle/configs/ball_dataset.yaml /content/repo/configs/ball_dataset.yaml
!ls /content/repo/data/raw/train/images | wc -l
!ls /content/repo/data/raw/valid/images | wc -l

In [ ]:
# Train. batch=16 (script default) fits comfortably in the T4's 16GB VRAM, vs batch=4
# on the local RTX 2050's 4GB - should be considerably faster wall-clock too.
# --project writes straight to Drive so checkpoints survive a runtime disconnect.
!mkdir -p /content/drive/MyDrive/tennis_colab/runs
!python scripts/train.py \
    --model weights/ball_detector.pt \
    --epochs 40 \
    --batch 16 \
    --patience 15 \
    --name ball_detector_v7_colab \
    --workers 2 \
    --project /content/drive/MyDrive/tennis_colab/runs

## After training

The best checkpoint is at `My Drive/tennis_colab/runs/ball_detector_v7_colab/weights/best.pt` in your Drive - no extra download step needed inside Colab, it's already there. On your local machine:

1. Download `best.pt` from that Drive folder.
2. `cp weights/ball_detector.pt weights/ball_detector_v7_backup.pt` (back up the current one first)
3. Move the downloaded `best.pt` to `weights/ball_detector.pt`
4. `python -m pytest tests/ -q` to sanity check nothing broke.